In [2]:
from pathlib import Path
import json
import xarray as xr
import gcsfs
import numpy as np
import pandas as pd
from tqdm import tqdm

mapping_path = Path("/tmp3/b12902101/Mazu/download_era5_data/variable_mapping.json")
with mapping_path.open() as f:
    mapping = json.load(f)

def _collect_vars(section):
    out = {}
    for target, vals in mapping.get(section, {}).items():
        src = vals.get("forecast_var")
        if src:
            out[src] = vals.get("data_root_var") or target
    return out

rename_map = {}
rename_map.update(_collect_vars("surface"))
rename_map.update(_collect_vars("atmospheric"))
needed_vars = sorted(rename_map.keys())

lat = slice(2.5, 41.25)
lon = slice(97.5, 147.25)
time = slice("2016-01-01T00:00:00.000000000", "2017-01-01T00:00:00.000000000")
# pred_time = slice(0, 12)
levels = [1000, 925, 850, 700, 500, 300, 150, 50]

fs = gcsfs.GCSFileSystem(token='anon')
mapper = fs.get_mapper("gs://weatherbench2/datasets/hres/2016-2022-0012-1440x721.zarr")
ds = xr.open_zarr(mapper, consolidated=True, decode_timedelta=False)
ds = ds.sel(
    latitude=lat,
    longitude=lon,
    time=time,
    level=levels,
    # prediction_timedelta=pred_time,
)
existing_vars = [v for v in needed_vars if v in ds.data_vars]
missing_vars = [v for v in needed_vars if v not in ds.data_vars]
if missing_vars:
    print("Missing forecast vars:", missing_vars)
ds = ds[existing_vars]
ds

<xarray.Dataset> Size: 165GB
Dimensions:                  (time: 733, prediction_timedelta: 41,
                              latitude: 156, longitude: 200, level: 8)
Coordinates:
  * time                     (time) datetime64[ns] 6kB 2016-01-01 ... 2017-01-01
  * prediction_timedelta     (prediction_timedelta) int64 328B 0 6 ... 234 240
  * latitude                 (latitude) float32 624B 2.5 2.75 3.0 ... 41.0 41.25
  * longitude                (longitude) float32 800B 97.5 97.75 ... 147.0 147.2
  * level                    (level) int32 32B 1000 925 850 700 500 300 150 50
Data variables:
    10m_u_component_of_wind  (time, prediction_timedelta, latitude, longitude) float32 4GB dask.array<chunksize=(1, 1, 156, 200), meta=np.ndarray>
    10m_v_component_of_wind  (time, prediction_timedelta, latitude, longitude) float32 4GB dask.array<chunksize=(1, 1, 156, 200), meta=np.ndarray>
    2m_temperature           (time, prediction_timedelta, latitude, longitude) float32 4GB dask.array<chunksize=(1, 1, 156, 200), meta=np.ndarray>
    geopotential             (time, prediction_timedelta, level, latitude, longitude) float32 30GB dask.array<chunksize=(1, 1, 8, 156, 200), meta=np.ndarray>
    mean_sea_level_pressure  (time, prediction_timedelta, latitude, longitude) float32 4GB dask.array<chunksize=(1, 1, 156, 200), meta=np.ndarray>
    specific_humidity        (time, prediction_timedelta, level, latitude, longitude) float32 30GB dask.array<chunksize=(1, 1, 8, 156, 200), meta=np.ndarray>
    temperature              (time, prediction_timedelta, level, latitude, longitude) float32 30GB dask.array<chunksize=(1, 1, 8, 156, 200), meta=np.ndarray>
    u_component_of_wind      (time, prediction_timedelta, level, latitude, longitude) float32 30GB dask.array<chunksize=(1, 1, 8, 156, 200), meta=np.ndarray>
    v_component_of_wind      (time, prediction_timedelta, level, latitude, longitude) float32 30GB dask.array<chunksize=(1, 1, 8, 156, 200), meta=np.ndarray>

In [8]:
ds.dims['time']

/tmp/ipykernel_1343135/4235786295.py:1: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  ds.dims['time']


733

In [9]:
out_root = Path("/tmp3/b12902101/era5_tw_forecast")
out_root.mkdir(parents=True, exist_ok=True)

def split_and_save_by_date(ds, out_root, time_values=None, limit=None):
    if time_values is None:
        time_values = ds["time"].values
    count = 0
    buckets = {}
    bar_format = "{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]"
    for init_time in tqdm(time_values, desc="Init time", unit="init", bar_format=bar_format):
        ds_step = ds.sel(time=[init_time])
        ds_step = ds_step.rename({
            k: v for k, v in rename_map.items() if k in ds_step.data_vars
        })

        upper_vars = [v for v in ds_step.data_vars if "level" in ds_step[v].dims]
        sfc_vars = [v for v in ds_step.data_vars if "level" not in ds_step[v].dims]
        ds_upper = ds_step[upper_vars].copy() if upper_vars else None
        ds_sfc = None
        if sfc_vars:
            ds_sfc = ds_step[sfc_vars].copy()
            ds_sfc = ds_sfc.drop_vars("level", errors="ignore")

        dt = pd.to_datetime(init_time)
        y = dt.strftime("%Y")
        ym = dt.strftime("%Y%m")
        ymd = dt.strftime("%Y%m%d")
        month_dir = out_root / y / ym
        month_dir.mkdir(parents=True, exist_ok=True)
        if ymd not in buckets:
            buckets[ymd] = {"dir": month_dir, "sfc": [], "upper": []}
        if ds_sfc is not None:
            buckets[ymd]["sfc"].append(ds_sfc)
        if ds_upper is not None:
            buckets[ymd]["upper"].append(ds_upper)

        count += 1
        if limit is not None and count >= limit:
            return _flush_buckets(buckets)

    _flush_buckets(buckets)

def _flush_buckets(buckets):
    for ymd, payload in buckets.items():
        month_dir = payload["dir"]
        if payload["sfc"]:
            ds_sfc_day = xr.concat(payload["sfc"], dim="time").sortby("time")
            ds_sfc_day.to_netcdf(month_dir / f"{ymd}_sfc.nc")
        if payload["upper"]:
            ds_upper_day = xr.concat(payload["upper"], dim="time").sortby("time")
            ds_upper_day.to_netcdf(month_dir / f"{ymd}_upper.nc")

# Example: limit to 2 steps for a quick sanity check
split_and_save_by_date(ds, out_root, limit=5)

Init time:   1%|          | 4/733 [00:33<1:41:22]


In [15]:
from pathlib import Path
import xarray as xr
out_root = Path("/tmp3/b12902101/era5_tw_forecast")

def check_daily_files(out_root, date_str, max_files=4):
    year = date_str[:4]
    ym = date_str[:6]
    month_dir = Path(out_root) / year / ym
    candidates = sorted(month_dir.glob(f"{date_str}_*.nc"))
    if not candidates:
        print(f"No files found for {date_str} in {month_dir}")
        return
    for path in candidates[:max_files]:
        ds_check = xr.open_dataset(path)
        dims = {k: int(v) for k, v in ds_check.dims.items()}
        vars_list = list(ds_check.data_vars)
        print(f"{path.name}: dims={dims}, vars={vars_list}")
        if "time" in ds_check.coords:
            print(f"  time[0]={ds_check['time'].values[0]}")
        ds_check.close()

# Example check: pick a date you exported
# check_daily_files(out_root, "20160101", max_files=2)
ds = xr.open_dataset(out_root / "2016" / "201601" / "20160101_upper.nc", decode_timedelta=False)

In [16]:
ds

<xarray.Dataset> Size: 30MB
Dimensions:               (time: 2, prediction_timedelta: 3, level: 8,
                           latitude: 156, longitude: 200)
Coordinates:
  * time                  (time) datetime64[ns] 16B 2016-01-01 2016-01-01T12:...
  * prediction_timedelta  (prediction_timedelta) int64 24B 0 6 12
  * level                 (level) int32 32B 1000 925 850 700 500 300 150 50
  * latitude              (latitude) float32 624B 2.5 2.75 3.0 ... 41.0 41.25
  * longitude             (longitude) float32 800B 97.5 97.75 ... 147.0 147.2
Data variables:
    z                     (time, prediction_timedelta, level, latitude, longitude) float32 6MB ...
    q                     (time, prediction_timedelta, level, latitude, longitude) float32 6MB ...
    t                     (time, prediction_timedelta, level, latitude, longitude) float32 6MB ...
    u                     (time, prediction_timedelta, level, latitude, longitude) float32 6MB ...
    v                     (time, prediction_timedelta, level, latitude, longitude) float32 6MB ...